# MNIST Digit Classification using Multiple Deep Learning APIs

Implementing the same CNN-based digit classifier using:
- Keras API
- PyTorch API
- JAX API
- TensorFlow API

Final comparison is based on training and validation accuracy.

In [4]:
# IMPORTING LIBRARIES:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

In [5]:
# LOADING DATASET:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

In [6]:
# NORMALIZATION & RESHAPING:
X_train = X_train / 255.0
X_test = X_test / 255.0

X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

In [7]:
# SPLITTING DATA:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

In [8]:
# ONE HOT ENCODING:
y_train = to_categorical(y_train, 10)
y_val = to_categorical(y_val, 10)
y_test = to_categorical(y_test, 10)

# **SECTION #1: KERAS MODEL**

In [9]:
# KERAS CNN(MODEL BUILDING):
from tensorflow.keras import layers, models

keras_model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
# COMPILING AND TRAINING:
keras_model.compile(optimizer='adam',
                    loss='categorical_crossentropy',
                    metrics=['accuracy'])

keras_history = keras_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=64
)

Epoch 1/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 44s 55ms/step - accuracy: 0.9395 - loss: 0.1952 - val_accuracy: 0.9818 - val_loss: 0.0625
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 82s 55ms/step - accuracy: 0.9826 - loss: 0.0570 - val_accuracy: 0.9867 - val_loss: 0.0434
Epoch 3/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 48s 64ms/step - accuracy: 0.9879 - loss: 0.0392 - val_accuracy: 0.9886 - val_loss: 0.0381
Epoch 4/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 75s 56ms/step - accuracy: 0.9907 - loss: 0.0288 - val_accuracy: 0.9875 - val_loss: 0.0401
Epoch 5/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 81s 54ms/step - accuracy: 0.9933 - loss: 0.0222 - val_accuracy: 0.9893 - val_loss: 0.0359


In [11]:
# EVALUATION:
keras_eval = keras_model.evaluate(X_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - accuracy: 0.9894 - loss: 0.0311


# **SECTION #2: PYTORCH MODEL**

In [12]:
# PYTORCH SETUP
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [13]:
# Convert numpy → torch tensors
X_train_t = torch.tensor(X_train).float().permute(0, 3, 1, 2)
X_val_t   = torch.tensor(X_val).float().permute(0, 3, 1, 2)
X_test_t  = torch.tensor(X_test).float().permute(0, 3, 1, 2)

y_train_t = torch.tensor(y_train.argmax(axis=1))
y_val_t   = torch.tensor(y_val.argmax(axis=1))
y_test_t  = torch.tensor(y_test.argmax(axis=1))

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=64)
test_loader  = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=64)

In [14]:
# BUILDING MODEL:
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 32, 3)
        self.conv2 = nn.Conv2d(32, 64, 3)

        self.fc1 = nn.Linear(64 * 5 * 5, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)

        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)

        x = x.reshape(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [15]:
# COMPILING:
model = CNN()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [16]:
# TRAINING:
epochs = 5

for epoch in range(epochs):
    model.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()


    print(f"Epoch {epoch+1}, Loss: {train_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.2594
Epoch 2, Loss: 0.0721
Epoch 3, Loss: 0.0494
Epoch 4, Loss: 0.0378
Epoch 5, Loss: 0.0288


In [17]:
correct = 0
total = 0

model.eval()
with torch.no_grad():
    for X_batch, y_batch in train_loader:
        outputs = model(X_batch)
        _, preds = torch.max(outputs, 1)

        total += y_batch.size(0)
        correct += (preds == y_batch).sum().item()

pt_train_acc = correct / total
print("PyTorch Train Accuracy:", pt_train_acc)

PyTorch Train Accuracy: 0.993


In [18]:
# EVALUATION:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        outputs = model(X_batch)
        _, predicted = torch.max(outputs, 1)

        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

pt_test_acc = correct / total
print("PyTorch Test Accuracy:", pt_test_acc)

PyTorch Test Accuracy: 0.9878


In [19]:
# VALIDATION:
correct = 0
total = 0

model.eval()
with torch.no_grad():
    for X_batch, y_batch in val_loader:
        outputs = model(X_batch)
        _, preds = torch.max(outputs, 1)

        total += y_batch.size(0)
        correct += (preds == y_batch).sum().item()

pt_val_acc = correct / total
print("PyTorch Val Accuracy:", pt_val_acc)

PyTorch Val Accuracy: 0.9865833333333334


# **SECTION #3: TENSORFLOW MODEL**

In [20]:
# MODEL BUILDING
import tensorflow as tf

tf_model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(28,28,1)),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10)
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [21]:
# COMPILATION:
tf_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [22]:
# MODEL TRAINING:
tf_history = tf_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=64
)

Epoch 1/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 44s 56ms/step - accuracy: 0.9412 - loss: 0.1901 - val_accuracy: 0.9803 - val_loss: 0.0630
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 84s 59ms/step - accuracy: 0.9830 - loss: 0.0537 - val_accuracy: 0.9849 - val_loss: 0.0494
Epoch 3/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 42s 55ms/step - accuracy: 0.9883 - loss: 0.0366 - val_accuracy: 0.9872 - val_loss: 0.0441
Epoch 4/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 42s 56ms/step - accuracy: 0.9916 - loss: 0.0273 - val_accuracy: 0.9893 - val_loss: 0.0363
Epoch 5/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 42s 56ms/step - accuracy: 0.9932 - loss: 0.0216 - val_accuracy: 0.9868 - val_loss: 0.0428


In [23]:
# EVALUATION:
tf_eval = tf_model.evaluate(X_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9879 - loss: 0.0358


# **SECTION #4: JAX MODEL**

In [24]:
# SETUP:
import jax
import jax.numpy as jnp
from flax import linen as flax_nn
from flax.training import train_state
import optax

In [25]:
# MODEL BUILDING:
class JAX_CNN(flax_nn.Module):
    @flax_nn.compact
    def __call__(self, x):
        x = flax_nn.Conv(32, (3,3))(x)
        x = flax_nn.relu(x)
        x = flax_nn.max_pool(x, (2,2))

        x = flax_nn.Conv(64, (3,3))(x)
        x = flax_nn.relu(x)
        x = flax_nn.max_pool(x, (2,2))

        x = x.reshape((x.shape[0], -1))

        x = flax_nn.Dense(128)(x)
        x = flax_nn.relu(x)
        x = flax_nn.Dense(64)(x)
        x = flax_nn.relu(x)
        x = flax_nn.Dense(10)(x)

        return x

In [26]:
# INITIALIZE THE MODEL:
jax_model = JAX_CNN()

rng = jax.random.PRNGKey(0)
sample_input = jnp.ones((1, 28, 28, 1))

params = jax_model.init(rng, sample_input)

In [27]:
# OPTIMIZER STATE:
state = train_state.TrainState.create(
    apply_fn=jax_model.apply,
    params=params,
    tx=optax.adam(1e-3)
)

In [28]:
# FORWARD & LOSS FUNCTION:
def loss_fn(params, batch):
    X, y = batch
    logits = jax_model.apply(params, X)
    loss = optax.softmax_cross_entropy_with_integer_labels(logits, y).mean()
    return loss

In [29]:
# TRAINING LOOP:
@jax.jit
def train_step(state, batch):
    grads = jax.grad(loss_fn)(state.params, batch)
    state = state.apply_gradients(grads=grads)
    return state

In [30]:
#TRAINING LOOP

X_train_j = jnp.array(X_train)
y_train_j = jnp.array(np.argmax(y_train, axis=1))

X_train_small = X_train[:10000]
y_train_small = y_train[:10000]

batch_size = 128
epochs = 3

y_train_labels = np.argmax(y_train_small, axis=1)

for epoch in range(epochs):

    perm = np.random.permutation(len(X_train_small))

    for i in range(0, len(X_train_small), batch_size):

        idx = perm[i:i+batch_size]

        X_batch = jnp.asarray(X_train_small[idx])
        y_batch = jnp.asarray(y_train_labels[idx])

        state = train_step(state, (X_batch, y_batch))

    print(f"Epoch {epoch+1} completed")

Epoch 1 completed
Epoch 2 completed
Epoch 3 completed


In [31]:
def predict(params, X):
    logits = jax_model.apply(params, X)
    return jnp.argmax(logits, axis=1)

preds = predict(state.params, jnp.array(X_test))
y_labels = jnp.argmax(y_test, axis=1)

jx_test_acc = (preds == y_labels).mean()

print(jx_test_acc)

0.9758


# **FINAL EVALUATION OF 4 MODELS:**

In [32]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score

# ==========================
# PYTORCH ACCURACY
# ==========================
model.eval()

with torch.no_grad():
    X_train_pt = torch.tensor(np.transpose(X_train, (0,3,1,2)), dtype=torch.float32)
    X_test_pt  = torch.tensor(np.transpose(X_test,  (0,3,1,2)), dtype=torch.float32)

    train_preds_pt = model(X_train_pt).argmax(dim=1).cpu().numpy()
    test_preds_pt  = model(X_test_pt).argmax(dim=1).cpu().numpy()

    y_train_true = np.argmax(y_train, axis=1)
    y_test_true  = np.argmax(y_test,  axis=1)

    pt_train_acc = accuracy_score(y_train_true, train_preds_pt)
    pt_test_acc  = accuracy_score(y_test_true,  test_preds_pt)

# Free PyTorch tensors immediately
del X_train_pt, X_test_pt
import gc; gc.collect()

# ==========================
# JAX ACCURACY — BATCHED
# ==========================
def predict_jax_batched(params, X_np, batch_size=512):
    all_preds = []
    for i in range(0, len(X_np), batch_size):
        batch = jnp.array(X_np[i:i+batch_size])
        logits = jax_model.apply(params, batch)
        preds  = jnp.argmax(logits, axis=1)
        all_preds.append(np.array(preds))
    return np.concatenate(all_preds)

jax_train_preds = predict_jax_batched(state.params, X_train)
jax_test_preds  = predict_jax_batched(state.params, X_test)

jx_train_acc = accuracy_score(y_train_true, jax_train_preds)
jx_test_acc  = accuracy_score(y_test_true,  jax_test_preds)

# ==========================
# FINAL COMPARISON TABLE
# ==========================
comparison = pd.DataFrame({
    "Framework":      ["Keras", "PyTorch", "TensorFlow", "JAX"],
    "Train Accuracy": [
        round(keras_history.history['accuracy'][-1],    4),
        round(pt_train_acc,                             4),
        round(tf_history.history['accuracy'][-1],       4),
        round(jx_train_acc,                             4),
    ],
    "Test Accuracy": [
        round(keras_history.history['val_accuracy'][-1], 4),
        round(pt_test_acc,                               4),
        round(tf_history.history['val_accuracy'][-1],    4),
        round(jx_test_acc,                               4),
    ]
})

print("\nCNN Framework Comparison\n")
print(comparison)
comparison


CNN Framework Comparison

    Framework  Train Accuracy  Test Accuracy
0       Keras          0.9933         0.9893
1     PyTorch          0.9930         0.9878
2  TensorFlow          0.9932         0.9868
3         JAX          0.9752         0.9758


,Framework,Train Accuracy,Test Accuracy
0,Keras,0.9933,0.9893
1,PyTorch,0.9930,0.9878
2,TensorFlow,0.9932,0.9868
3,JAX,0.9752,0.9758
